# Qwen3-Omni 하이햇·라이드 마디 판정 — GPU 서버 실행 (S15P31A307-325)

11/09에 돌린 오디오 LLM 시험에서 Qwen3-Omni만 `audioread` 패키지가 없어 실패했다. 이 노트북은 고친 스크립트로 Qwen3-Omni 하나만 다시 돌린다.

- 하는 일: 작은 시험 120마디를 원곡·분리 드럼으로 한 번씩 묻고(240번), 통과 기준을 넘는 오디오가 있으면 3,513마디 전체로 이어서 묻는다
- 이미 서버에 있는 것: 마디 소리 파일·문제·정답(`~/ssaplash/cymrole_audio/`), omni 가상환경, Qwen3-Omni 가중치. 그래서 바로 판정부터 시작한다
- 결과: `~/ssaplash/results/cymrole_audio.zip` (11/09에 돌린 다른 모델 답도 같이 들어감)

**준비물.** 이 파일만 JupyterHub `ssaplash/notebooks/`에 올린다. zip은 올리지 않는다. 스크립트 3개가 1번 셀 안에 들어 있다.

**실행.** 1번 셀을 한 번 실행하고, 2번 셀로 진행을 본다. 브라우저를 닫아도 계속 돈다.


In [ ]:
# 1. 시작 (여러 번 실행해도 됨). 고친 스크립트 3개를 쓰고 Qwen3-Omni만 백그라운드로 시작함
#    이미 돌고 있으면 새로 시작하지 않음. 끝난 단계는 건너뛰고, 실패한 단계만 다시 함
import os
import subprocess
import sys
from pathlib import Path

GPU_ID = "6"                                   # 지정받은 GPU
WORK = Path.home() / "ssaplash"
CA = WORK / "cymrole_audio"                    # 11/09에 만든 작업 폴더

# 고친 스크립트. 저장소 AI/scripts의 같은 이름 파일과 내용이 같음
FILES = {
    "cymrole_audio_llm.py": "\"\"\"오디오 LLM 하나로 마디마다 하이햇·라이드 판정 (11/09, 설계 CHART-003-audio-llm-cymbal, S15P31A307-325).\n\ncases.jsonl의 마디 wav를 모델에 들려주고 H(하이햇)·R(라이드)·N(둘 다 아님) 중 한 글자를 묻는다.\n첫 답 토큰 위치의 H·R·N 토큰 확률을 합 1로 맞춰 answers_<백엔드>_<오디오>.jsonl에 한 줄씩 덧붙임.\n이미 답한 마디는 건너뜀(끊겨도 다시 실행하면 이어 감). 정답 파일은 읽지 않음.\n백엔드마다 모델 카드·공식 저장소 예제의 불러오기·입력 방식을 따름. 생성은 탐욕(do_sample=False).\n\n  python scripts/cymrole_audio_llm.py --backend qwen3omni --cases <폴더> --audio mix drums --subset <폴더>/pilot.txt --out <폴더>/answers\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport sys\nimport time\nfrom pathlib import Path\n\nQUESTION = (\n    \"This audio clip is one bar from a rock or pop song. Listen to the drums. \"\n    \"Which cymbal is the drummer playing repeatedly to keep time in this bar? \"\n    \"Answer with exactly one letter: H if it is the hi-hat, R if it is the ride cymbal, N if neither is heard.\"\n)\nLETTERS = (\"H\", \"R\", \"N\")\nMAX_NEW = 8\nTOPK = 200  # 첫 토큰 확률 상위 몇 개에서 H·R·N 토큰을 찾을지\n\n\ndef letter_of(tok: str) -> str | None:\n    s = tok.strip().strip(\"\\\"'*.:()[]`\").upper()\n    return s if s in LETTERS else None\n\n\ndef letter_probs(logits, decode_one) -> tuple[dict[str, float], float]:\n    \"\"\"첫 답 위치 로짓(1차원)에서 H·R·N 토큰 확률. 공백이 붙은 토큰도 같은 글자로 합침. (합 1 확률, 원래 질량).\"\"\"\n    import torch\n\n    p = torch.softmax(logits.float().reshape(-1), dim=-1)\n    top = torch.topk(p, min(TOPK, p.numel()))\n    acc = dict.fromkeys(LETTERS, 0.0)\n    for v, i in zip(top.values.tolist(), top.indices.tolist(), strict=True):\n        lab = letter_of(decode_one(int(i)))\n        if lab is not None:\n            acc[lab] += float(v)\n    mass = sum(acc.values())\n    return ({k: v / mass for k, v in acc.items()} if mass > 0 else acc), mass\n\n\ndef letter_from_text(text: str) -> str | None:\n    \"\"\"확률을 못 얻었을 때 답 글에서 판정. 첫 글자, 아니면 낱말.\"\"\"\n    t = text.strip()\n    if t and letter_of(t.split()[0]) is not None:\n        return letter_of(t.split()[0])\n    low = t.lower()\n    if re.search(r\"\\bride\\b\", low):\n        return \"R\"\n    if re.search(r\"hi-?\\s?hat\", low):\n        return \"H\"\n    if re.search(r\"\\b(neither|none)\\b\", low):\n        return \"N\"\n    return None\n\n\ndef first_text_step(steps, decode_one):\n    \"\"\"직접 만든 생성 루프에서 기록한 (로짓, 고른 토큰) 중 글자가 처음 나온 단계의 로짓.\"\"\"\n    for logits, tok in steps:\n        if decode_one(int(tok)).strip():\n            return logits\n    return None\n\n\n# ---------------------------------------------------------------- 백엔드\nclass Backend:\n    \"\"\"load()로 모델을 올리고 ask(wav 경로)가 (첫 답 위치 로짓 또는 None, 답 글, 토큰 하나 디코드 함수)를 돌려줌.\"\"\"\n\n    repo_id = \"\"\n\n    def __init__(self, args):\n        self.args = args\n\n    def snapshot(self, repo_id: str | None = None) -> str:\n        from huggingface_hub import snapshot_download\n\n        return snapshot_download(repo_id or self.repo_id)\n\n    def load(self) -> None:\n        raise NotImplementedError\n\n    def ask(self, wav: str):\n        raise NotImplementedError\n\n\ndef _gen_first_logits(out):\n    \"\"\"generate(return_dict_in_generate) 결과의 첫 생성 위치 로짓. output_logits가 있으면 처리 전 로짓.\"\"\"\n    seq = getattr(out, \"logits\", None) or getattr(out, \"scores\", None)\n    return seq[0][0] if seq else None\n\n\nclass Qwen3Omni(Backend):\n    repo_id = \"Qwen/Qwen3-Omni-30B-A3B-Instruct\"\n\n    def load(self):\n        import torch\n        from transformers import Qwen3OmniMoeForConditionalGeneration, Qwen3OmniMoeProcessor\n\n        self.torch = torch\n        self.model = Qwen3OmniMoeForConditionalGeneration.from_pretrained(\n            self.repo_id, dtype=\"auto\", device_map=\"auto\", attn_implementation=self.args.attn\n        )\n        self.model.disable_talker()  # 음성 출력 끔\n        self.warned = False\n        self.warned_shape = False\n        self.proc = Qwen3OmniMoeProcessor.from_pretrained(self.repo_id)\n\n    def ask(self, wav):\n        from qwen_omni_utils import process_mm_info\n\n        conv = [\n            {\n                \"role\": \"user\",\n                \"content\": [{\"type\": \"audio\", \"audio\": wav}, {\"type\": \"text\", \"text\": QUESTION}],\n            }\n        ]\n        text = self.proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)\n        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)\n        inputs = self.proc(text=text, audio=audios, images=images, videos=videos, return_tensors=\"pt\",\n                           padding=True, use_audio_in_video=False)  # fmt: skip\n        inputs = inputs.to(self.model.device).to(self.model.dtype)\n        with self.torch.no_grad():\n            # 모델 카드 배치 예제와 같은 인자 + 첫 토큰 점수. thinker_ 접두 인자가 안 먹히면\n            # 카드 인자만으로 다시 생성하고 점수는 아래 thinker forward로 구함\n            card = dict(\n                return_audio=False, use_audio_in_video=False, thinker_return_dict_in_generate=True\n            )\n            # 11/10 서버: 카드 예제는 (글, 소리) 둘을 돌려준다고 보고 둘로 나눠 받지만, 서버 transformers는\n            # 소리를 끄면 생성 결과 하나만 돌려줘서 나눠 받다가 결과의 항목 이름이 들어갔음. 둘 다 받게 함\n            try:\n                out = self.model.generate(\n                    **inputs, **card, thinker_output_scores=True,\n                    thinker_max_new_tokens=MAX_NEW, thinker_do_sample=False,\n                )  # fmt: skip\n            except (TypeError, ValueError) as e:\n                if not self.warned:\n                    print(f\"[llm] qwen3omni 점수 인자 거절, 카드 인자로 다시: {e}\", flush=True)\n                    self.warned = True\n                out = self.model.generate(**inputs, **card, thinker_max_new_tokens=MAX_NEW)\n            text_ids = out[0] if isinstance(out, tuple) else out\n            seqs = getattr(\n                text_ids, \"sequences\", text_ids\n            )  # 결과 묶음이면 sequences, 텐서면 그대로\n            logits = _gen_first_logits(text_ids)\n            if logits is None:  # 점수가 안 넘어오면 thinker 한 번 forward\n                logits = self.model.thinker(**inputs).logits[0, -1]\n        if not self.warned_shape:\n            print(f\"[llm] qwen3omni 생성 결과 {type(out).__name__}, 점수 {'있음' if _gen_first_logits(text_ids) is not None else '없음, forward로 구함'}\", flush=True)  # fmt: skip\n            self.warned_shape = True\n        n = inputs[\"input_ids\"].shape[1]\n        ans = self.proc.batch_decode(seqs[:, n:], skip_special_tokens=True,\n                                     clean_up_tokenization_spaces=False)[0]  # fmt: skip\n        return logits, ans, lambda i: self.proc.tokenizer.decode([i])\n\n\nclass _HFGenerate(Backend):\n    \"\"\"processor 입력을 model.generate로 넘기는 모델 공통.\"\"\"\n\n    def generate(self, inputs, n_prompt, decode_batch, decode_one):\n        with self.torch.no_grad():\n            out = self.model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False, output_scores=True,\n                                      output_logits=True, return_dict_in_generate=True)  # fmt: skip\n        ans = decode_batch(out.sequences[:, n_prompt:])[0]\n        return _gen_first_logits(out), ans, decode_one\n\n\nclass Voxtral(_HFGenerate):\n    repo_id = \"mistralai/Voxtral-Small-24B-2507\"\n\n    def load(self):\n        import torch\n        from transformers import AutoProcessor, VoxtralForConditionalGeneration\n\n        self.torch = torch\n        self.proc = AutoProcessor.from_pretrained(self.repo_id)\n        self.model = VoxtralForConditionalGeneration.from_pretrained(\n            self.repo_id, torch_dtype=torch.bfloat16, device_map=\"cuda\"\n        )\n\n    def ask(self, wav):\n        conv = [\n            {\n                \"role\": \"user\",\n                \"content\": [{\"type\": \"audio\", \"path\": wav}, {\"type\": \"text\", \"text\": QUESTION}],\n            }\n        ]\n        inputs = self.proc.apply_chat_template(conv).to(\"cuda\", dtype=self.torch.bfloat16)\n        return self.generate(\n            inputs, inputs.input_ids.shape[1],\n            lambda s: self.proc.batch_decode(s, skip_special_tokens=True),\n            lambda i: self.proc.tokenizer.decode([i]),\n        )  # fmt: skip\n\n\nclass AFNext(_HFGenerate):\n    repo_id = \"nvidia/audio-flamingo-next-hf\"\n\n    def load(self):\n        import torch\n        import transformers\n        from transformers import AutoModel, AutoProcessor\n\n        self.torch = torch\n        self.proc = AutoProcessor.from_pretrained(self.repo_id)\n        # 11/09 서버: AutoModel이 언어 모델 머리 없는 MusicFlamingoModel을 올려 generate가 없었음.\n        # 모델 카드가 말하는 생성용 클래스를 먼저 찾고, 없으면 같은 계열 생성용 클래스\n        names = (\"AudioFlamingoNextForConditionalGeneration\", \"MusicFlamingoForConditionalGeneration\",\n                 \"AudioFlamingo3ForConditionalGeneration\")  # fmt: skip\n        cls = next((getattr(transformers, n) for n in names if hasattr(transformers, n)), AutoModel)\n        print(f\"[llm] afnext 모델 클래스 {cls.__name__}\", flush=True)\n        self.model = cls.from_pretrained(\n            self.repo_id, torch_dtype=torch.bfloat16, device_map=\"auto\"\n        ).eval()\n\n    def ask(self, wav):\n        conv = [\n            [\n                {\n                    \"role\": \"user\",\n                    \"content\": [{\"type\": \"text\", \"text\": QUESTION}, {\"type\": \"audio\", \"path\": wav}],\n                }\n            ]\n        ]\n        batch = self.proc.apply_chat_template(conv, tokenize=True, add_generation_prompt=True,\n                                              return_dict=True).to(self.model.device)  # fmt: skip\n        if \"input_features\" in batch:\n            batch[\"input_features\"] = batch[\"input_features\"].to(self.model.dtype)\n        return self.generate(\n            batch, batch[\"input_ids\"].shape[1],\n            lambda s: self.proc.batch_decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=False),\n            lambda i: self.proc.tokenizer.decode([i]),\n        )  # fmt: skip\n\n\nclass Nemotron(_HFGenerate):\n    repo_id = \"nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16\"\n\n    def load(self):\n        import torch\n        from transformers import AutoModel, AutoProcessor\n\n        self.torch = torch\n        self.proc = AutoProcessor.from_pretrained(self.repo_id, trust_remote_code=True)\n        # 11/09 서버: 설정 기본값이 flash_attention_2라 flash-attn이 없는 환경에서 불러오기 실패. sdpa로 바꿈\n        self.model = AutoModel.from_pretrained(self.repo_id, trust_remote_code=True, torch_dtype=torch.bfloat16,\n                                               device_map=\"cuda\", attn_implementation=\"sdpa\").eval()  # fmt: skip\n\n    def ask(self, wav):\n        import soundfile as sf\n\n        audio, sr = sf.read(wav, dtype=\"float32\")\n        conv = [\n            {\"role\": \"user\", \"content\": [{\"type\": \"audio\"}, {\"type\": \"text\", \"text\": QUESTION}]}\n        ]\n        prompt = self.proc.apply_chat_template(conv, tokenize=False, add_generation_prompt=True,\n                                               enable_thinking=False)  # fmt: skip\n        inputs = self.proc(\n            text=[prompt], audio=[audio], audio_kwargs={\"sampling_rate\": sr}, return_tensors=\"pt\"\n        )\n        inputs = {k: (v.to(\"cuda\") if hasattr(v, \"to\") else v) for k, v in inputs.items()}\n        return self.generate(\n            inputs, inputs[\"input_ids\"].shape[1],\n            lambda s: self.proc.tokenizer.batch_decode(s, skip_special_tokens=True),\n            lambda i: self.proc.tokenizer.decode([i]),\n        )  # fmt: skip\n\n\nclass MossMusic(_HFGenerate):\n    \"\"\"공식 저장소(OpenMOSS/MOSS-Music) infer.py 방식. --repo에 저장소 경로.\"\"\"\n\n    repo_id = \"OpenMOSS-Team/MOSS-Music-8B-Instruct\"\n\n    def load(self):\n        import torch\n\n        sys.path.insert(0, str(self.args.repo))\n        from src.modeling_moss_music import MossMusicModel\n        from src.processing_moss_music import MossMusicProcessor\n\n        self.torch = torch\n        path = self.snapshot()\n        self.model = MossMusicModel.from_pretrained(path, trust_remote_code=True, torch_dtype=\"auto\",\n                                                    device_map=\"cuda:0\").eval()  # fmt: skip\n        self.proc = MossMusicProcessor.from_pretrained(\n            path, trust_remote_code=True, enable_time_marker=True\n        )\n\n    def ask(self, wav):\n        # 저장소 load_audio(torchaudio.load)와 같은 결과(모노 1차원 numpy, mel_sr)를 soundfile로 만듦.\n        # torchaudio 2.9의 load는 ffmpeg 공유 라이브러리가 필요한데 서버에는 정적 ffmpeg만 있음\n        import soundfile as sf\n        import torchaudio\n\n        raw, sr = sf.read(wav, dtype=\"float32\")\n        raw = raw.mean(axis=1) if raw.ndim == 2 else raw\n        if sr != self.proc.config.mel_sr:\n            t = torchaudio.functional.resample(\n                self.torch.from_numpy(raw), sr, self.proc.config.mel_sr\n            )\n            raw = t.numpy()\n        inputs = self.proc(text=QUESTION, audios=[raw], return_tensors=\"pt\").to(self.model.device)\n        if inputs.get(\"audio_data\") is not None:\n            inputs[\"audio_data\"] = inputs[\"audio_data\"].to(self.model.dtype)\n        inputs[\"audio_input_mask\"] = inputs[\"input_ids\"] == self.proc.audio_token_id\n        return self.generate(\n            inputs, inputs[\"input_ids\"].shape[1],\n            lambda s: [self.proc.decode(s[0], skip_special_tokens=True)],\n            lambda i: self.proc.decode([i]),\n        )  # fmt: skip\n\n\nclass StepAudio2Mini(Backend):\n    \"\"\"공식 저장소(stepfun-ai/Step-Audio2) stepaudio2.py의 StepAudio2.__call__과 같은 입력, 점수만 더 받음.\"\"\"\n\n    repo_id = \"stepfun-ai/Step-Audio-2-mini\"\n\n    def load(self):\n        import torch\n\n        sys.path.insert(0, str(self.args.repo))\n        from stepaudio2 import StepAudio2\n\n        self.torch = torch\n        self.m = StepAudio2(self.snapshot())\n\n    def ask(self, wav):\n        from transformers import GenerationConfig\n\n        m, torch = self.m, self.torch\n        messages = [\n            {\"role\": \"system\", \"content\": \"You are an expert in audio analysis, please analyze the audio content and answer the questions accurately.\"},\n            {\"role\": \"human\", \"content\": [{\"type\": \"audio\", \"audio\": wav}, {\"type\": \"text\", \"text\": QUESTION}]},\n            {\"role\": \"assistant\", \"content\": None},\n        ]  # fmt: skip\n        msgs, mels = m.apply_chat_template(messages)\n        ids = []\n        for msg in msgs:\n            if isinstance(msg, str):\n                ids.append(\n                    m.llm_tokenizer(text=msg, return_tensors=\"pt\", padding=True)[\"input_ids\"]\n                )\n            else:\n                ids.append(torch.tensor([msg], dtype=torch.int32))\n        prompt = torch.cat(ids, dim=-1).cuda()\n        from utils import padding_mels\n\n        mels_t, mel_lens = padding_mels(mels)\n        cfg = GenerationConfig(max_new_tokens=MAX_NEW, pad_token_id=m.llm_tokenizer.pad_token_id,\n                               eos_token_id=m.eos_token_id, do_sample=False, output_scores=True,\n                               output_logits=True, return_dict_in_generate=True)  # fmt: skip\n        with torch.no_grad():\n            out = m.llm.generate(input_ids=prompt, wavs=mels_t.cuda(), wav_lens=mel_lens.cuda(),\n                                 attention_mask=torch.ones_like(prompt), generation_config=cfg,\n                                 tokenizer=m.llm_tokenizer)  # fmt: skip\n        toks = [\n            i for i in out.sequences[0, prompt.shape[-1] :].tolist() if i < 151688\n        ]  # 151688부터 음성 토큰\n        return (\n            _gen_first_logits(out),\n            m.llm_tokenizer.decode(toks),\n            lambda i: m.llm_tokenizer.decode([i]),\n        )\n\n\nclass KimiAudio7B(Backend):\n    \"\"\"공식 저장소(MoonshotAI/Kimi-Audio) KimiAudio.generate(output_type='text'). 글 로짓은 샘플러를 감싸 기록.\"\"\"\n\n    repo_id = \"moonshotai/Kimi-Audio-7B-Instruct\"\n\n    def load(self):\n        sys.path.insert(0, str(self.args.repo))\n        import kimia_infer.utils.sampler as ks\n        from kimia_infer.api.kimia import KimiAudio\n\n        self.steps: list = []\n        orig = ks.KimiASampler.sample_text_logits\n        rec = self.steps\n\n        def sample_text_logits(sampler, logits, recent_tokens=None):\n            tok = orig(sampler, logits, recent_tokens=recent_tokens)\n            lg = logits[:, -1] if logits.dim() == 3 else logits\n            rec.append((lg[0].float().cpu(), int(tok.reshape(-1)[0])))\n            return tok\n\n        ks.KimiASampler.sample_text_logits = sample_text_logits\n        self.m = KimiAudio(model_path=self.snapshot(), load_detokenizer=False)\n        self.offset = self.m.kimia_token_offset\n        self.special = {self.m.extra_tokens.kimia_text_eos, self.m.extra_tokens.kimia_text_blank}\n\n    def ask(self, wav):\n        self.steps.clear()\n        msgs = [{\"role\": \"user\", \"message_type\": \"text\", \"content\": QUESTION},\n                {\"role\": \"user\", \"message_type\": \"audio\", \"content\": wav}]  # fmt: skip\n        _, text = self.m.generate(msgs, output_type=\"text\", text_temperature=0.0, text_top_k=5,\n                                  max_new_tokens=32)  # fmt: skip\n        tok = self.m.prompt_manager.text_tokenizer\n\n        def decode_one(i):\n            return \"\" if i >= self.offset or i in self.special else tok.decode([i])\n\n        return first_text_step(self.steps, decode_one), text, decode_one\n\n\nclass MiMoAudio7B(Backend):\n    \"\"\"공식 저장소(XiaomiMiMo/MiMo-Audio) audio_understanding_sft. 글 샘플러를 탐욕·기록용으로 바꿈.\"\"\"\n\n    repo_id = \"XiaomiMiMo/MiMo-Audio-7B-Instruct\"\n\n    def load(self):\n        sys.path.insert(0, str(self.args.repo))\n        from src.mimo_audio.mimo_audio import MimoAudio\n        from src.mimo_audio.modeling_mimo_audio import MiMoSampler\n\n        self.steps: list = []\n        rec = self.steps\n\n        class Recording(MiMoSampler):\n            def sample(self, scores, removed_tokens=None):\n                raw = scores[0].float().cpu().clone()\n                tok = super().sample(scores, removed_tokens=removed_tokens)\n                rec.append((raw, int(tok.reshape(-1)[0])))\n                return tok\n\n        self.m = MimoAudio(self.snapshot(), self.snapshot(\"XiaomiMiMo/MiMo-Audio-Tokenizer\"))\n        self.m.task_sampler_configs[\"audio_understanding\"][\"global\"] = Recording(do_sample=False, temperature=1.0,\n                                                                                  top_p=1.0)  # fmt: skip\n\n    def ask(self, wav):\n        self.steps.clear()\n        text = self.m.audio_understanding_sft(wav, QUESTION, thinking=False)\n\n        def decode_one(i):\n            return self.m.tokenizer.decode([i], skip_special_tokens=True)\n\n        return first_text_step(self.steps, decode_one), str(text), decode_one\n\n\nclass GeminiGMS(Backend):\n    \"\"\"GMS 프록시로 Gemini 3.8 Flash interactions API 호출 (11/10 사용자 지시, 로컬 실행).\n\n    interactions API는 토큰 확률을 주지 않아서 글자와 0~100 확신도를 같이 답하게 함.\n    고른 글자에 확신도, 나머지 두 글자에 남은 몫을 반씩 줌. 이 모델은 생각 끄기(minimal)를 받지 않아 low로 둠.\n    키는 환경 변수 GMS_KEY, 없으면 AI/.env의 GMS_KEY 줄에서 읽고 화면에 찍지 않음.\n    \"\"\"\n\n    URL = \"https://gms.ssafy.io/gmsapi/generativelanguage.googleapis.com/v1beta/interactions\"\n    MODEL = \"gemini-3.8-flash\"\n    PROMPT = (\n        \"This audio clip is one bar from a rock or pop song. Listen to the drums. Which cymbal is the drummer \"\n        \"playing repeatedly to keep time in this bar? Reply with one letter (H = hi-hat, R = ride cymbal, \"\n        \"N = neither heard), a space, and your confidence from 0 to 100. Example: R 70\"\n    )\n\n    def load(self):\n        import os\n\n        key = os.environ.get(\"GMS_KEY\")\n        if not key:\n            env = Path(__file__).resolve().parents[1] / \".env\"\n            for line in env.read_text(encoding=\"utf-8\").splitlines():\n                if line.startswith(\"GMS_KEY=\"):\n                    key = line.split(\"=\", 1)[1].strip().strip('\"').strip(\"'\")\n        if not key:\n            raise SystemExit(\"GMS_KEY가 없음 (환경 변수 또는 AI/.env)\")\n        self.key = key\n        self.usage = {\"input\": 0, \"thought\": 0, \"output\": 0, \"calls\": 0}\n\n    def call(self, wav: str) -> dict:\n        import base64\n        import urllib.error\n        import urllib.request\n\n        body = {\n            \"model\": self.MODEL,\n            \"input\": [\n                {\"type\": \"text\", \"text\": self.PROMPT},\n                {\n                    \"type\": \"audio\",\n                    \"data\": base64.b64encode(Path(wav).read_bytes()).decode(),\n                    \"mime_type\": \"audio/wav\",\n                },\n            ],\n            \"generation_config\": {\"thinking_level\": \"low\", \"max_output_tokens\": 1024},\n        }\n        data = json.dumps(body).encode()\n        for attempt in range(5):  # 429·5xx는 기다렸다 다시\n            req = urllib.request.Request(\n                self.URL,\n                data=data,\n                headers={\"Content-Type\": \"application/json\", \"x-goog-api-key\": self.key},\n            )\n            try:\n                return json.loads(urllib.request.urlopen(req, timeout=120).read())\n            except urllib.error.HTTPError as e:\n                if e.code in (429, 500, 502, 503, 504) and attempt < 4:\n                    time.sleep(5 * 2**attempt)\n                    continue\n                raise SystemExit(f\"GMS HTTP {e.code}: {e.read().decode()[:300]}\") from e\n            except urllib.error.URLError:\n                if attempt < 4:\n                    time.sleep(5 * 2**attempt)\n                    continue\n                raise\n        raise SystemExit(\"GMS 호출 실패\")\n\n    def ask(self, wav):\n        r = self.call(wav)\n        u = r.get(\"usage\") or {}\n        self.usage[\"input\"] += int(u.get(\"total_input_tokens\", 0))\n        self.usage[\"thought\"] += int(u.get(\"total_thought_tokens\", 0))\n        self.usage[\"output\"] += int(u.get(\"total_output_tokens\", 0))\n        self.usage[\"calls\"] += 1\n        # 마디마다 답 파일에 남겨 끊겨도 사용량 합계를 낼 수 있게 함\n        self.last_usage = {\n            \"input\": int(u.get(\"total_input_tokens\", 0)),\n            \"thought\": int(u.get(\"total_thought_tokens\", 0)),\n            \"output\": int(u.get(\"total_output_tokens\", 0)),\n            \"status\": r.get(\"status\"),\n        }\n        text = \"\".join(\n            c.get(\"text\", \"\")\n            for st in (r.get(\"steps\") or [])\n            if st.get(\"type\") == \"model_output\"\n            for c in st.get(\"content\", [])\n            if c.get(\"type\") == \"text\"\n        )\n        return stated_probs(text), text, None\n\n\ndef stated_probs(text: str) -> dict[str, float] | None:\n    \"\"\"'R 70' 꼴 답을 확률로. 고른 글자에 확신도, 나머지 둘에 남은 몫을 반씩. 형식이 안 맞으면 None.\"\"\"\n    m = re.match(r\"\\s*([HRN])\\b\\W*(\\d{1,3})\", text.strip().upper())\n    if not m:\n        return None\n    c = min(100, int(m.group(2))) / 100\n    return {k: (c if k == m.group(1) else (1 - c) / 2) for k in LETTERS}\n\n\nBACKENDS = {\n    \"qwen3omni\": Qwen3Omni,\n    \"nemotron\": Nemotron,\n    \"voxtral\": Voxtral,\n    \"afnext\": AFNext,\n    \"mossmusic\": MossMusic,\n    \"mimoaudio\": MiMoAudio7B,\n    \"stepaudio2\": StepAudio2Mini,\n    \"kimiaudio\": KimiAudio7B,\n    \"gemini\": GeminiGMS,\n}\n\n\nclass Fake(Backend):\n    \"\"\"테스트용 가짜 백엔드. 파일 이름 길이로 정해진 로짓을 냄.\"\"\"\n\n    def load(self):\n        pass\n\n    def ask(self, wav):\n        import torch\n\n        k = len(Path(wav).name) % 3\n        logits = torch.full((5,), -5.0)\n        logits[k] = 3.0\n        names = [\"H\", \" R\", \"N\", \"x\", \"y\"]\n        return logits, names[k].strip(), lambda i: names[i]\n\n\nBACKENDS[\"fake\"] = Fake\n\n\n# ---------------------------------------------------------------- 실행\ndef answer_row(cid: str, logits, text: str, decode_one, sec: float) -> dict:\n    src = \"logits\"\n    if isinstance(logits, dict):  # 모델이 말로 밝힌 확신도로 만든 확률 (gemini)\n        p, mass, src = logits, 1.0, \"stated\"\n    elif logits is not None:\n        p, mass = letter_probs(logits, decode_one)\n    else:\n        p, mass = dict.fromkeys(LETTERS, 0.0), 0.0\n    if mass <= 0:\n        lab = letter_from_text(text)\n        p = {k: float(k == lab) for k in LETTERS} if lab else dict.fromkeys(LETTERS, 1 / 3)\n        src = \"text\" if lab else \"none\"\n    choice = max(LETTERS, key=lambda k: p[k])\n    return {\"case_id\": cid, \"p\": p, \"choice\": choice, \"p_mass\": round(mass, 4), \"p_source\": src,\n            \"raw\": text[:200], \"sec\": round(sec, 3)}  # fmt: skip\n\n\ndef run(args) -> int:\n    cases = [\n        json.loads(x)\n        for x in (args.cases / \"cases.jsonl\").read_text(encoding=\"utf-8\").splitlines()\n        if x.strip()\n    ]\n    if args.subset is not None:\n        keep = {\n            x.strip() for x in args.subset.read_text(encoding=\"utf-8\").splitlines() if x.strip()\n        }\n        cases = [c for c in cases if c[\"case_id\"] in keep]\n    if args.limit:\n        cases = cases[: args.limit]\n    args.out.mkdir(parents=True, exist_ok=True)\n    be = None\n    for audio in args.audio:  # 한 번 불러온 모델로 오디오 종류를 차례로\n        be = run_audio(args, cases, audio, be)\n    return 0\n\n\ndef run_audio(args, cases: list[dict], audio: str, be):\n    name = f\"{args.backend}_{audio}\"\n    out = args.out / f\"answers_{name}.jsonl\"\n    done = set()\n    if out.is_file():\n        done = {\n            json.loads(x)[\"case_id\"]\n            for x in out.read_text(encoding=\"utf-8\").splitlines()\n            if x.strip()\n        }\n    todo = [c for c in cases if c[\"case_id\"] not in done]\n    print(f\"[llm] {name}: 전체 {len(cases)}, 이미 답함 {len(done)}, 남음 {len(todo)}\", flush=True)\n    if not todo:\n        return be\n    load_sec = 0.0\n    if be is None:\n        be = BACKENDS[args.backend](args)\n        t0 = time.time()\n        be.load()\n        load_sec = time.time() - t0\n        print(f\"[llm] {name}: 불러오기 {load_sec:.0f}초\", flush=True)\n    times = []\n    with out.open(\"a\", encoding=\"utf-8\") as fh:\n        for k, c in enumerate(todo):\n            wav = str((args.cases / c[f\"audio_{audio}\"]).resolve())\n            t = time.time()\n            logits, text, decode_one = be.ask(wav)\n            sec = time.time() - t\n            times.append(sec)\n            row = answer_row(c[\"case_id\"], logits, text, decode_one, sec)\n            if getattr(be, \"last_usage\", None) is not None:  # API 백엔드의 이 마디 토큰 수\n                row[\"usage\"] = be.last_usage\n            fh.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n            fh.flush()\n            if k + 1 == 10 or (k + 1) % 50 == 0:\n                print(\n                    f\"[llm] {name}: {k + 1}/{len(todo)}, 마디당 {sum(times) / len(times):.2f}초\",\n                    flush=True,\n                )\n    meta = {\"name\": name, \"load_sec\": round(load_sec, 1), \"n\": len(times),\n            \"sec_per_bar\": round(sum(times) / max(1, len(times)), 3), \"peak_gpu_gb\": peak_gpu_gb(),\n            \"usage\": getattr(be, \"usage\", None)}  # fmt: skip\n    (args.out / f\"meta_{name}.json\").write_text(\n        json.dumps(meta, ensure_ascii=False), encoding=\"utf-8\"\n    )\n    print(f\"[llm] {name}: 끝 {meta}\", flush=True)\n    return be\n\n\ndef peak_gpu_gb() -> float | None:\n    try:\n        import torch\n\n        return (\n            round(torch.cuda.max_memory_allocated() / 1e9, 1) if torch.cuda.is_available() else None\n        )\n    except Exception:  # noqa: BLE001  # torch가 없는 환경\n        return None\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument(\"--backend\", choices=sorted(BACKENDS), required=True)\n    ap.add_argument(\"--cases\", type=Path, required=True, help=\"cases.jsonl·clips가 있는 폴더\")\n    ap.add_argument(\"--audio\", nargs=\"+\", choices=(\"mix\", \"drums\"), required=True)\n    ap.add_argument(\"--subset\", type=Path, default=None, help=\"case_id 목록 파일. 없으면 전체\")\n    ap.add_argument(\"--out\", type=Path, required=True)\n    ap.add_argument(\n        \"--repo\",\n        type=Path,\n        default=None,\n        help=\"공식 저장소 경로 (mossmusic·stepaudio2·kimiaudio·mimoaudio)\",\n    )\n    ap.add_argument(\"--attn\", default=\"sdpa\", help=\"Qwen3-Omni attention 구현\")\n    ap.add_argument(\"--limit\", type=int, default=0)\n    return run(ap.parse_args())\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
    "cymrole_audio_eval.py": "\"\"\"오디오 LLM 하이햇·라이드 마디 판정 채점 (11/09, 설계 CHART-003-audio-llm-cymbal, S15P31A307-325).\n\nscore: 답 파일(answers_<모델>_<오디오>.jsonl)마다 정확도·균형 정확도·AUC·고친 마디·망친 마디를 계산하고\n  같은 마디의 지금 채보(current)·8패드 모델(bar_pad)·분리 트랙 에너지(bar_stem)와 나란히 csv로 씀.\n  --gate면 통과 조합 목록을 json으로 씀(AUC GATE_AUC 이상, 비교 기준 AUC보다 GATE_MARGIN 이상 높음).\nsections: 확신 기준을 PICK_SPLITS 마디로 고르고, 대상 split의 답을 judge_cymrole.py score가 읽는\n  곡별 sections JSON으로 바꿈.\n서버에서도 돌아야 해서 numpy만 씀(mlflow는 --mlflow일 때만 불러옴).\n\n  python scripts/cymrole_audio_eval.py score --answers data/results/cymrole_audio/answers \\\n      --truth data/results/cymrole_audio/truth.jsonl --subset data/results/cymrole_audio/pilot.txt \\\n      --out data/results/cymrole_audio/pilot_scores.csv --gate data/results/cymrole_audio/gate.json\n  python scripts/cymrole_audio_eval.py sections --answer data/results/cymrole_audio/answers/answers_qwen3omni_mix.jsonl \\\n      --truth data/results/cymrole_audio/truth.jsonl --split hold8 --out data/results/cymrole_audio/sections/qwen3omni_mix\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nGATE_AUC = 0.80\nGATE_MARGIN = 0.05\nPICK_SPLITS = (\"val10\", \"hold30\")\nBASELINES = (\"current\", \"bar_pad\", \"bar_stem\")\n\n\ndef read_jsonl(path: Path) -> list[dict]:\n    return [json.loads(x) for x in path.read_text(encoding=\"utf-8\").splitlines() if x.strip()]\n\n\ndef auc(score: np.ndarray, pos: np.ndarray) -> float:\n    \"\"\"양성(pos=True)이 음성보다 점수가 높을 확률. 같은 점수는 0.5로 셈 (Mann-Whitney).\"\"\"\n    p, n = score[pos], score[~pos]\n    if not len(p) or not len(n):\n        return float(\"nan\")\n    order = np.argsort(np.concatenate([p, n]), kind=\"mergesort\")\n    allv = np.concatenate([p, n])[order]\n    ranks = np.empty(len(allv))\n    i = 0\n    while i < len(allv):\n        j = i\n        while j + 1 < len(allv) and allv[j + 1] == allv[i]:\n            j += 1\n        ranks[i : j + 1] = (i + j) / 2 + 1\n        i = j + 1\n    r = np.empty(len(allv))\n    r[order] = ranks\n    return float((r[: len(p)].sum() - len(p) * (len(p) + 1) / 2) / (len(p) * len(n)))\n\n\ndef baseline_answer(t: dict, name: str) -> tuple[str, float]:\n    \"\"\"비교 기준을 (판정, 라이드 쪽 점수)로. 점수가 없으면 0.\"\"\"\n    if name == \"current\":\n        c = t[\"current\"]\n        return c, {\"R\": 1.0, \"H\": -1.0}.get(c, 0.0)\n    v = t[name]\n    v = 0.0 if v is None else float(v)\n    return (\"R\" if v > 0 else \"H\" if v < 0 else \"N\"), v\n\n\ndef metrics(choice: list[str], score: list[float], truths: list[dict]) -> dict:\n    \"\"\"정답 H·R 마디에서 판정 지표. N 판정은 틀린 것으로 셈.\"\"\"\n    y = np.array([t[\"truth\"] for t in truths])\n    c = np.array(choice)\n    cur_ok = np.array([t[\"current\"] == t[\"truth\"] for t in truths])\n    ok = c == y\n    acc_r = float(np.mean(ok[y == \"R\"])) if np.any(y == \"R\") else float(\"nan\")\n    acc_h = float(np.mean(ok[y == \"H\"])) if np.any(y == \"H\") else float(\"nan\")\n    return {\n        \"n\": len(y),\n        \"acc\": float(np.mean(ok)),\n        \"bal_acc\": (acc_r + acc_h) / 2,\n        \"acc_R\": acc_r,\n        \"acc_H\": acc_h,\n        \"auc\": auc(np.array(score, float), y == \"R\"),\n        \"n_current_wrong\": int(np.sum(~cur_ok)),\n        \"fixed\": int(np.sum(ok & ~cur_ok)),\n        \"broken\": int(np.sum(~ok & cur_ok)),\n        \"answered_N\": int(np.sum(c == \"N\")),\n    }\n\n\ndef model_score(a: dict) -> float:\n    p = a[\"p\"]\n    return float(p.get(\"R\", 0.0)) - float(p.get(\"H\", 0.0))\n\n\ndef cmd_score(args) -> int:\n    truth = {t[\"case_id\"]: t for t in read_jsonl(args.truth)}\n    subset = None\n    if args.subset is not None:\n        subset = [\n            x.strip() for x in args.subset.read_text(encoding=\"utf-8\").splitlines() if x.strip()\n        ]\n    rows = []\n    files = sorted(Path(args.answers).glob(\"answers_*.jsonl\"))\n    for f in files:\n        ans = {a[\"case_id\"]: a for a in read_jsonl(f)}\n        ids = [c for c in (subset or ans) if c in ans and c in truth]\n        if not ids:\n            continue\n        ts = [truth[c] for c in ids]\n        m = metrics([ans[c][\"choice\"] for c in ids], [model_score(ans[c]) for c in ids], ts)\n        name = f.stem.removeprefix(\"answers_\")\n        secs = [float(ans[c].get(\"sec\", 0.0)) for c in ids]\n        rows.append(\n            {\n                \"name\": name,\n                **m,\n                \"sec_per_bar\": float(np.mean(secs)),\n                \"missing\": len(subset or ans) - len(ids),\n            }\n        )\n        # 같은 마디의 비교 기준\n        for b in BASELINES:\n            pairs = [baseline_answer(t, b) for t in ts]\n            mb = metrics([p[0] for p in pairs], [p[1] for p in pairs], ts)\n            rows.append({\"name\": f\"{name}|{b}\", **mb, \"sec_per_bar\": 0.0, \"missing\": 0})\n    if not rows:\n        print(\"[eval] 채점할 답이 없음\", flush=True)\n        return 1\n    args.out.parent.mkdir(parents=True, exist_ok=True)\n    with args.out.open(\"w\", newline=\"\", encoding=\"utf-8\") as fh:\n        w = csv.DictWriter(fh, fieldnames=list(rows[0]))\n        w.writeheader()\n        w.writerows(rows)\n    for r in rows:\n        if \"|\" not in r[\"name\"]:\n            print(f\"[eval] {r['name']}: n {r['n']} 균형 정확도 {r['bal_acc']:.3f} AUC {r['auc']:.3f} \"\n                  f\"고친 {r['fixed']}/{r['n_current_wrong']} 망친 {r['broken']} N답 {r['answered_N']}\", flush=True)  # fmt: skip\n    if args.gate is not None:\n        passed = gate(rows)\n        args.gate.write_text(json.dumps(passed, ensure_ascii=False, indent=1), encoding=\"utf-8\")\n        print(f\"[eval] 통과 {len(passed)}개: {', '.join(passed) or '없음'}\", flush=True)\n    if args.mlflow:\n        log_mlflow(rows, args)\n    return 0\n\n\ndef gate(rows: list[dict]) -> list[str]:\n    \"\"\"AUC GATE_AUC 이상이고 같은 마디의 bar_pad·bar_stem AUC보다 GATE_MARGIN 이상 높은 답 이름.\"\"\"\n    by = {r[\"name\"]: r for r in rows}\n    out = []\n    for r in rows:\n        if \"|\" in r[\"name\"] or not np.isfinite(r[\"auc\"]):\n            continue\n        base = max(\n            (\n                by[f\"{r['name']}|{b}\"][\"auc\"]\n                for b in (\"bar_pad\", \"bar_stem\")\n                if np.isfinite(by[f\"{r['name']}|{b}\"][\"auc\"])\n            ),\n            default=0.5,\n        )\n        if r[\"auc\"] >= GATE_AUC and r[\"auc\"] >= base + GATE_MARGIN:\n            out.append(r[\"name\"])\n    return out\n\n\ndef log_mlflow(rows: list[dict], args) -> None:\n    import mlflow\n\n    mlflow.set_experiment(\"chart-cymrole-audio\")\n    for r in rows:\n        if \"|\" in r[\"name\"]:\n            continue\n        model, _, audio = r[\"name\"].rpartition(\"_\")\n        with mlflow.start_run(run_name=f\"{r['name']}_{args.out.stem}\"):\n            mlflow.set_tags(\n                {\"jira\": \"S15P31A307-325\", \"model\": model, \"audio\": audio, \"subset\": args.out.stem}\n            )\n            mlflow.log_metrics(\n                {k: float(v) for k, v in r.items() if k != \"name\" and np.isfinite(float(v))}\n            )\n            for b in BASELINES:\n                rb = next(x for x in rows if x[\"name\"] == f\"{r['name']}|{b}\")\n                mlflow.log_metrics({f\"{b}_auc\": rb[\"auc\"], f\"{b}_bal_acc\": rb[\"bal_acc\"]})\n            mlflow.log_artifact(str(Path(args.answers) / f\"answers_{r['name']}.jsonl\"))\n            mlflow.log_artifact(str(args.out))\n\n\ndef pick_threshold(ans: dict, truth: dict) -> float:\n    \"\"\"PICK_SPLITS 마디에서 '망친 마디 − 고친 마디'가 가장 작은 확신 기준. 같으면 높은 기준.\"\"\"\n    ids = [c for c, t in truth.items() if t[\"split\"] in PICK_SPLITS and c in ans]\n    best_t, best = 1.01, 0\n    for th in np.round(np.arange(0.50, 1.0001, 0.02), 2):\n        score = 0\n        for c in ids:\n            a, t = ans[c], truth[c]\n            if a[\"choice\"] not in (\"H\", \"R\") or max(a[\"p\"].values()) < th:\n                continue\n            score += int(a[\"choice\"] != t[\"truth\"] and t[\"current\"] == t[\"truth\"])\n            score -= int(a[\"choice\"] == t[\"truth\"] and t[\"current\"] != t[\"truth\"])\n        if score < best or (score == best and th > best_t and best_t <= 1.0):\n            best_t, best = float(th), score\n    return best_t\n\n\ndef to_sections(ans: dict, truth: dict, split: str, th: float) -> dict[str, dict]:\n    \"\"\"대상 split의 확신 기준 이상 H·R 답을 곡별 sections JSON으로.\"\"\"\n    out: dict[str, dict] = {}\n    for c, t in truth.items():\n        if t[\"split\"] != split or c not in ans:\n            continue\n        a = ans[c]\n        conf = max(a[\"p\"].values())\n        if a[\"choice\"] not in (\"H\", \"R\") or conf < th:\n            continue\n        out.setdefault(t[\"song\"], {\"sections\": []})[\"sections\"].append(\n            {\n                \"from_bar\": t[\"bar\"],\n                \"to_bar\": t[\"bar\"],\n                \"timekeeping\": a[\"choice\"],\n                \"confidence\": conf,\n            }\n        )\n    return out\n\n\ndef cmd_sections(args) -> int:\n    truth = {t[\"case_id\"]: t for t in read_jsonl(args.truth)}\n    ans = {a[\"case_id\"]: a for a in read_jsonl(args.answer)}\n    th = args.threshold if args.threshold is not None else pick_threshold(ans, truth)\n    secs = to_sections(ans, truth, args.split, th)\n    args.out.mkdir(parents=True, exist_ok=True)\n    for song, body in secs.items():\n        (args.out / f\"{song}.json\").write_text(\n            json.dumps(body, ensure_ascii=False, indent=1), encoding=\"utf-8\"\n        )\n    n = sum(len(b[\"sections\"]) for b in secs.values())\n    print(\n        f\"[eval] 확신 기준 {th:.2f}, {args.split} 곡 {len(secs)}개 마디 {n}개를 sections로 씀\",\n        flush=True,\n    )\n    return 0\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    sub = ap.add_subparsers(dest=\"cmd\", required=True)\n    s = sub.add_parser(\"score\")\n    s.add_argument(\"--answers\", type=Path, required=True, help=\"answers_*.jsonl이 든 폴더\")\n    s.add_argument(\"--truth\", type=Path, required=True)\n    s.add_argument(\n        \"--subset\", type=Path, default=None, help=\"채점할 case_id 목록 파일 (작은 시험 pilot.txt)\"\n    )\n    s.add_argument(\"--out\", type=Path, required=True)\n    s.add_argument(\"--gate\", type=Path, default=None)\n    s.add_argument(\"--mlflow\", action=\"store_true\")\n    c = sub.add_parser(\"sections\")\n    c.add_argument(\"--answer\", type=Path, required=True)\n    c.add_argument(\"--truth\", type=Path, required=True)\n    c.add_argument(\"--split\", default=\"hold8\")\n    c.add_argument(\"--threshold\", type=float, default=None, help=\"없으면 PICK_SPLITS로 고름\")\n    c.add_argument(\"--out\", type=Path, required=True)\n    args = ap.parse_args()\n    return cmd_score(args) if args.cmd == \"score\" else cmd_sections(args)\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
    "cymrole_audio_pipeline.py": "\"\"\"서버에서 오디오 LLM 하이햇·라이드 판정을 끝까지 돌림 (11/09, 설계 CHART-003-audio-llm-cymbal, S15P31A307-325).\n\n단계: check → 모델마다 [가상환경 → 공식 저장소 → 작은 시험(원곡·분리 드럼) → 통과 판단 → 통과한 오디오 전체 마디]\n→ 결과 묶기. 단계가 끝나면 done/<단계> 파일을 남기고, 다시 실행하면 끝난 단계는 건너뜀.\n모델 하나가 실패하면 failed/<단계>에 오류를 남기고 다음 모델로 넘어감(다시 실행하면 다시 시도).\n운영 코드 폴더·운영 .venv는 건드리지 않음. 가상환경·가중치·저장소는 전부 작업 폴더 아래에 둠.\n이 스크립트는 표준 라이브러리만 씀.\n\n  nohup python scripts/cymrole_audio_pipeline.py run --work ~/ssaplash/cymrole_audio >> pipeline.log 2>&1 &\n  python scripts/cymrole_audio_pipeline.py status --work ~/ssaplash/cymrole_audio\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport zipfile\nfrom pathlib import Path\n\nTORCH_CU128 = \"https://download.pytorch.org/whl/cu128\"\nTORCH_CU124 = \"https://download.pytorch.org/whl/cu124\"\nFA_WHEEL = (\"https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/\"\n            \"flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-{py}-{py}-linux_x86_64.whl\")  # fmt: skip\n\n# 가상환경: (파이썬, 설치 명령 목록). 명령의 {repo}는 그 환경이 쓰는 공식 저장소 경로\nENVS = {\n    \"omni\": (\n        \"3.12\",\n        [\n            [\n                \"torch==2.9.1\",\n                \"torchaudio==2.9.1\",\n                \"torchvision==0.24.1\",\n                \"--index-url\",\n                TORCH_CU128,\n            ],\n            [\n                \"transformers\",\n                \"accelerate\",\n                \"qwen-omni-utils\",\n                \"audioread\",  # qwen-omni-utils가 맨 위에서 불러오지만 의존 목록에 없음\n                \"mistral-common[audio]\",\n                \"soundfile\",\n                \"librosa\",\n                \"numpy\",\n                \"huggingface_hub[hf_xet]\",\n                \"timm\",\n                \"einops\",\n            ],\n        ],\n    ),\n    \"moss\": (\n        \"3.12\",\n        [\n            [\"-e\", \"{repo}[torch-runtime]\", \"--extra-index-url\", TORCH_CU128],\n            [\"soundfile\", \"huggingface_hub[hf_xet]\"],\n        ],\n    ),\n    \"mimo\": (\n        \"3.12\",\n        [\n            [\"torch==2.6.0\", \"torchaudio==2.6.0\", \"--index-url\", TORCH_CU124],\n            [\"-r\", \"{repo}/requirements.txt\", \"--extra-index-url\", TORCH_CU124],\n            [FA_WHEEL.format(py=\"cp312\")],\n            [\"soundfile\", \"huggingface_hub[hf_xet]\"],\n        ],\n    ),\n    \"step\": (\n        \"3.12\",\n        [\n            [\"torch==2.6.0\", \"torchaudio==2.6.0\", \"--index-url\", TORCH_CU124],\n            [\n                \"transformers==4.49.0\",\n                \"accelerate\",\n                \"librosa\",\n                \"soundfile\",\n                \"einops\",\n                \"huggingface_hub[hf_xet]\",\n            ],\n        ],\n    ),\n    \"kimi\": (\n        \"3.10\",\n        [\n            [\"torch==2.6.0\", \"torchaudio==2.6.0\", \"--index-url\", TORCH_CU124],\n            # 11/09 서버: portalocker가 PyTorch 저장소에도 있어 uv가 그쪽만 보다가 버전을 못 찾음\n            [\n                \"-r\",\n                \"{work}/kimi_requirements.txt\",\n                \"--extra-index-url\",\n                TORCH_CU124,\n                \"--index-strategy\",\n                \"unsafe-best-match\",\n            ],\n            [FA_WHEEL.format(py=\"cp310\")],\n        ],\n    ),\n}\n# 가상환경을 만든 뒤 알게 된 빠진 패키지. 판정 전마다 설치함(이미 있으면 바로 끝남)\nEXTRAS = {\"omni\": [\"audioread\"]}  # qwen-omni-utils가 맨 위에서 불러오지만 의존 목록에 없음\nREPOS = {\n    \"moss\": \"https://github.com/OpenMOSS/MOSS-Music.git\",\n    \"mimo\": \"https://github.com/XiaomiMiMo/MiMo-Audio.git\",\n    \"step\": \"https://github.com/stepfun-ai/Step-Audio2.git\",\n    \"kimi\": \"https://github.com/MoonshotAI/Kimi-Audio.git\",\n}\n# 백엔드: (가상환경, 가중치 저장소 목록). 순서대로 실행\nMODELS = {\n    \"qwen3omni\": (\"omni\", [\"Qwen/Qwen3-Omni-30B-A3B-Instruct\"]),\n    \"nemotron\": (\"omni\", [\"nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-BF16\"]),\n    \"voxtral\": (\"omni\", [\"mistralai/Voxtral-Small-24B-2507\"]),\n    \"afnext\": (\"omni\", [\"nvidia/audio-flamingo-next-hf\"]),\n    \"mossmusic\": (\"moss\", [\"OpenMOSS-Team/MOSS-Music-8B-Instruct\"]),\n    \"mimoaudio\": (\"mimo\", [\"XiaomiMiMo/MiMo-Audio-7B-Instruct\", \"XiaomiMiMo/MiMo-Audio-Tokenizer\"]),\n    \"stepaudio2\": (\"step\", [\"stepfun-ai/Step-Audio-2-mini\"]),\n    \"kimiaudio\": (\"kimi\", [\"moonshotai/Kimi-Audio-7B-Instruct\", \"THUDM/glm-4-voice-tokenizer\"]),\n    \"fake\": (\"omni\", []),  # 로컬 시험용 가짜 백엔드\n}\nREAL_MODELS = [m for m in MODELS if m != \"fake\"]\nAUDIOS = (\"mix\", \"drums\")\n\n\nclass Ctx:\n    def __init__(self, args):\n        self.a = args\n        self.work = Path(args.work).expanduser().resolve()\n        self.done = self.work / \"done\"\n        self.failed = self.work / \"failed\"\n        self.logs = self.work / \"log\"\n        self.answers = self.work / \"answers\"\n        for d in (self.done, self.failed, self.logs, self.answers):\n            d.mkdir(parents=True, exist_ok=True)\n        self.env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(args.gpu), HF_HOME=str(self.work / \"hf_cache\"),\n                        PYTHONUNBUFFERED=\"1\", PYTHONIOENCODING=\"utf-8\", TOKENIZERS_PARALLELISM=\"false\")  # fmt: skip\n\n    def py(self, env: str) -> str:\n        if self.a.python:  # 로컬 시험: 가상환경 대신 지정한 파이썬\n            return self.a.python\n        return str(self.work / \"envs\" / env / \"bin\" / \"python\")\n\n\ndef log(msg: str) -> None:\n    print(f\"[{time.strftime('%m-%d %H:%M:%S')}] {msg}\", flush=True)\n\n\ndef sh(ctx: Ctx, cmd: list[str], logname: str, cwd: Path | None = None) -> None:\n    \"\"\"명령을 돌려 출력은 log/<logname>.log에 덧붙임. 실패하면 CalledProcessError.\"\"\"\n    with (ctx.logs / f\"{logname}.log\").open(\"a\", encoding=\"utf-8\") as fh:\n        fh.write(f\"\\n$ {' '.join(cmd)}\\n\")\n        fh.flush()\n        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, check=True, env=ctx.env, cwd=cwd)\n\n\ndef stage(ctx: Ctx, name: str, fn) -> bool:\n    \"\"\"done/<name>이 있으면 건너뜀. 성공하면 done, 실패하면 failed/<name>에 오류를 쓰고 False.\"\"\"\n    if (ctx.done / name).exists():\n        return True\n    log(f\"시작 {name}\")\n    t = time.time()\n    try:\n        fn()\n    except Exception as e:  # noqa: BLE001  # 모델 하나가 실패해도 다음 모델로\n        (ctx.failed / name).write_text(f\"{type(e).__name__}: {e}\\n\", encoding=\"utf-8\")\n        log(f\"실패 {name}: {type(e).__name__}: {e} (log 폴더 확인)\")\n        return False\n    (ctx.failed / name).unlink(missing_ok=True)\n    (ctx.done / name).write_text(time.strftime(\"%Y-%m-%d %H:%M:%S\"), encoding=\"utf-8\")\n    log(f\"끝 {name} ({(time.time() - t) / 60:.1f}분)\")\n    return True\n\n\ndef gpu_free_gb(gpu: int) -> float | None:\n    try:\n        out = subprocess.run([\"nvidia-smi\", \"--query-gpu=index,memory.free\", \"--format=csv,noheader,nounits\"],\n                             capture_output=True, text=True, check=True).stdout  # fmt: skip\n    except Exception:  # noqa: BLE001\n        return None\n    for line in out.splitlines():\n        idx, free = (x.strip() for x in line.split(\",\"))\n        if int(idx) == gpu:\n            return float(free) / 1024\n    return None\n\n\ndef disk_free_gb(path: Path) -> float:\n    return shutil.disk_usage(path).free / 1e9\n\n\ndef check(ctx: Ctx) -> None:\n    info = {\"gpu\": ctx.a.gpu, \"gpu_free_gb\": gpu_free_gb(ctx.a.gpu), \"disk_free_gb\": round(disk_free_gb(ctx.work), 1),\n            \"uv\": shutil.which(ctx.a.uv.split()[0]) is not None, \"git\": shutil.which(\"git\") is not None,\n            \"cases\": sum(1 for _ in (ctx.work / \"cases.jsonl\").open(encoding=\"utf-8\")),\n            \"pilot\": sum(1 for _ in (ctx.work / \"pilot.txt\").open(encoding=\"utf-8\"))}  # fmt: skip\n    (ctx.work / \"check.json\").write_text(\n        json.dumps(info, ensure_ascii=False, indent=1), encoding=\"utf-8\"\n    )\n    log(f\"점검 {info}\")\n    if not ctx.a.python and (not info[\"uv\"] or not info[\"git\"]):\n        raise RuntimeError(\"uv 또는 git이 없음\")\n\n\ndef make_repo(ctx: Ctx, env: str) -> None:\n    dest = ctx.work / \"repos\" / env\n    if dest.exists():\n        shutil.rmtree(dest)\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    sh(ctx, [\"git\", \"clone\", \"--depth\", \"1\", REPOS[env], str(dest)], f\"repo_{env}\")\n    if env == \"kimi\":  # 추론에 안 쓰는 학습용(deepspeed)과 따로 받는 flash_attn은 빼고 설치\n        req = (dest / \"requirements.txt\").read_text(encoding=\"utf-8\").splitlines()\n        keep = [r for r in req if r.strip() and not r.startswith((\"flash_attn\", \"deepspeed\"))]\n        (ctx.work / \"kimi_requirements.txt\").write_text(\"\\n\".join(keep) + \"\\n\", encoding=\"utf-8\")\n\n\ndef make_env(ctx: Ctx, env: str) -> None:\n    pyver, steps = ENVS[env]\n    path = ctx.work / \"envs\" / env\n    if path.exists():\n        shutil.rmtree(path)\n    uv = ctx.a.uv.split()\n    sh(ctx, [*uv, \"venv\", \"--python\", pyver, str(path)], f\"env_{env}\")\n    repo = str(ctx.work / \"repos\" / env)\n    for pkgs in steps:\n        args = [p.format(repo=repo, work=str(ctx.work)) for p in pkgs]\n        sh(ctx, [*uv, \"pip\", \"install\", \"--python\", ctx.py(env), *args], f\"env_{env}\")\n    sh(\n        ctx,\n        [ctx.py(env), \"-c\", \"import torch;print(torch.__version__, torch.cuda.is_available())\"],\n        f\"env_{env}\",\n    )\n\n\ndef download(ctx: Ctx, model: str) -> None:\n    env, repos = MODELS[model]\n    for r in repos:\n        sh(ctx, [ctx.py(env), \"-c\", f\"from huggingface_hub import snapshot_download;print(snapshot_download('{r}'))\"],\n           f\"download_{model}\")  # fmt: skip\n\n\ndef ask(ctx: Ctx, model: str, audios: list[str], subset: Path | None) -> None:\n    env, _ = MODELS[model]\n    if not ctx.a.python and EXTRAS.get(env):\n        sh(\n            ctx,\n            [*ctx.a.uv.split(), \"pip\", \"install\", \"--python\", ctx.py(env), *EXTRAS[env]],\n            f\"env_{env}\",\n        )\n    cmd = [ctx.py(env), str(ctx.work / \"scripts\" / \"cymrole_audio_llm.py\"), \"--backend\", model,\n           \"--cases\", str(ctx.work), \"--audio\", *audios, \"--out\", str(ctx.answers)]  # fmt: skip\n    if subset is not None:\n        cmd += [\"--subset\", str(subset)]\n    if env in REPOS:\n        cmd += [\"--repo\", str(ctx.work / \"repos\" / env)]\n    sh(ctx, cmd, f\"ask_{model}\")\n\n\ndef score_pilot(ctx: Ctx, model: str) -> list[str]:\n    \"\"\"이 모델 작은 시험 답만 채점하고 통과한 오디오 목록을 돌려줌.\"\"\"\n    one = ctx.work / \"pilot_answers\" / model\n    one.mkdir(parents=True, exist_ok=True)\n    for a in AUDIOS:\n        f = ctx.answers / f\"answers_{model}_{a}.jsonl\"\n        if f.is_file():\n            pilot = set((ctx.work / \"pilot.txt\").read_text(encoding=\"utf-8\").split())\n            rows = [\n                x\n                for x in f.read_text(encoding=\"utf-8\").splitlines()\n                if json.loads(x)[\"case_id\"] in pilot\n            ]\n            (one / f.name).write_text(\"\\n\".join(rows) + \"\\n\", encoding=\"utf-8\")\n    gate = ctx.work / \"gate\" / f\"{model}.json\"\n    gate.parent.mkdir(parents=True, exist_ok=True)\n    sh(ctx, [ctx.py(\"omni\"), str(ctx.work / \"scripts\" / \"cymrole_audio_eval.py\"), \"score\", \"--answers\", str(one),\n             \"--truth\", str(ctx.work / \"truth.jsonl\"), \"--subset\", str(ctx.work / \"pilot.txt\"),\n             \"--out\", str(ctx.work / \"pilot_scores\" / f\"{model}.csv\"), \"--gate\", str(gate)], f\"score_{model}\")  # fmt: skip\n    passed = json.loads(gate.read_text(encoding=\"utf-8\"))\n    return [n.removeprefix(f\"{model}_\") for n in passed]\n\n\ndef free_weights(ctx: Ctx, model: str) -> None:\n    \"\"\"디스크가 모자랄 때만 이 모델 가중치를 지움.\"\"\"\n    if disk_free_gb(ctx.work) >= ctx.a.min_disk_gb:\n        return\n    for r in MODELS[model][1]:\n        d = ctx.work / \"hf_cache\" / \"hub\" / (\"models--\" + r.replace(\"/\", \"--\"))\n        if d.exists():\n            shutil.rmtree(d)\n            log(f\"디스크 {disk_free_gb(ctx.work):.0f}GB 남아 {r} 가중치를 지움\")\n\n\ndef run_model(ctx: Ctx, model: str) -> None:\n    env, _ = MODELS[model]\n    if not ctx.a.python:\n        if env in REPOS and not stage(ctx, f\"repo_{env}\", lambda: make_repo(ctx, env)):\n            return\n        if not stage(ctx, f\"env_{env}\", lambda: make_env(ctx, env)):\n            return\n        if not stage(ctx, f\"download_{model}\", lambda: download(ctx, model)):\n            return\n    if not stage(\n        ctx, f\"pilot_{model}\", lambda: ask(ctx, model, list(AUDIOS), ctx.work / \"pilot.txt\")\n    ):\n        return\n    if not ctx.a.no_full:\n        if not ctx.a.python and not (ctx.done / \"env_omni\").exists():  # 채점은 omni 환경의 numpy로\n            stage(ctx, \"env_omni\", lambda: make_env(ctx, \"omni\"))\n        holder: dict = {}\n        if not stage(ctx, f\"gate_{model}\", lambda: holder.setdefault(\"p\", score_pilot(ctx, model))):\n            return\n        passed = holder.get(\"p\")\n        if passed is None:\n            g = ctx.work / \"gate\" / f\"{model}.json\"\n            passed = [\n                n.removeprefix(f\"{model}_\") for n in json.loads(g.read_text(encoding=\"utf-8\"))\n            ]\n        log(f\"{model} 작은 시험 통과 오디오: {passed or '없음'}\")\n        if passed and not stage(ctx, f\"full_{model}\", lambda: ask(ctx, model, passed, None)):\n            return\n    free_weights(ctx, model)\n\n\ndef pack(ctx: Ctx) -> None:\n    out = Path(ctx.a.results).expanduser() / \"cymrole_audio.zip\"\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(out, \"w\", zipfile.ZIP_DEFLATED) as z:\n        for sub in (\"answers\", \"log\", \"gate\", \"pilot_scores\", \"failed\", \"done\"):\n            for p in sorted((ctx.work / sub).rglob(\"*\")):\n                if p.is_file():\n                    z.write(p, f\"cymrole_audio/{p.relative_to(ctx.work)}\")\n        if (ctx.work / \"check.json\").is_file():\n            z.write(ctx.work / \"check.json\", \"cymrole_audio/check.json\")\n    log(f\"결과 zip {out} {out.stat().st_size / 1e6:.1f}MB\")\n\n\ndef pid_alive(pid: int) -> bool:\n    \"\"\"그 번호의 프로세스가 이 파이프라인으로 실제로 돌고 있는지.\n\n    11/10 서버: 끝난 파이프라인이 좀비(상태 Z)로 남아 os.kill(pid, 0)만으로는 '도는 중'으로 보였음.\n    리눅스면 /proc에서 상태가 Z가 아니고 명령줄에 이 스크립트 이름이 있을 때만 도는 중으로 봄.\n    \"\"\"\n    proc = Path(f\"/proc/{pid}\")\n    if Path(\"/proc\").is_dir():\n        try:\n            state = (proc / \"stat\").read_text().rsplit(\")\", 1)[1].split()[0]\n            cmd = (proc / \"cmdline\").read_bytes()\n        except OSError:\n            return False\n        return state != \"Z\" and b\"cymrole_audio_pipeline\" in cmd\n    try:\n        os.kill(pid, 0)\n    except OSError:\n        return False\n    return True\n\n\ndef status(work: Path) -> None:\n    \"\"\"진행 확인: 돌고 있는지, 단계별 끝·실패, 답 개수, 로그 끝.\"\"\"\n    pid_file = work / \"pipeline.pid\"\n    alive = False\n    if pid_file.is_file():\n        pid = int(pid_file.read_text().strip())\n        alive = pid_alive(pid)\n        print(f\"파이프라인 {'도는 중' if alive else '멈춤 또는 끝'} (pid {pid})\")\n    done = {p.name for p in (work / \"done\").glob(\"*\")} if (work / \"done\").is_dir() else set()\n    failed = {p.name: p.read_text(encoding=\"utf-8\").strip() for p in (work / \"failed\").glob(\"*\")}         if (work / \"failed\").is_dir() else {}  # fmt: skip\n    for m in REAL_MODELS:\n        env = MODELS[m][0]\n        names = ([f\"repo_{env}\"] if env in REPOS else []) + [\n            f\"env_{env}\",\n            f\"download_{m}\",\n            f\"pilot_{m}\",\n            f\"gate_{m}\",\n            f\"full_{m}\",\n        ]\n        marks = \" \".join(\n            f\"{n.split('_')[0]}:{'끝' if n in done else '실패' if n in failed else '-'}\"\n            for n in names\n        )\n        print(f\"  {m:11s} {marks}\")\n    for n, e in failed.items():\n        print(f\"  실패 {n}: {e[:200]}\")\n    ans = work / \"answers\"\n    if ans.is_dir():\n        for f in sorted(ans.glob(\"answers_*.jsonl\")):\n            print(\n                f\"  답 {f.stem.removeprefix('answers_')}: {sum(1 for _ in f.open(encoding='utf-8'))}마디\"\n            )\n    gate_dir = work / \"gate\"\n    if gate_dir.is_dir():\n        for g in sorted(gate_dir.glob(\"*.json\")):\n            print(\n                f\"  작은 시험 통과 {g.stem}: {json.loads(g.read_text(encoding='utf-8')) or '없음'}\"\n            )\n    logf = work / \"pipeline.log\"\n    if logf.is_file():\n        lines = logf.read_text(encoding=\"utf-8\", errors=\"replace\").splitlines()\n        print(\"파이프라인 로그 끝:\")\n        for x in lines[-4:]:\n            print(\"   \", x)\n        cur = [x for x in lines if \"시작 \" in x]\n        if alive and cur:\n            name = cur[-1].split(\"시작 \", 1)[1].strip()\n            sub = name.replace(\"pilot_\", \"ask_\").replace(\"full_\", \"ask_\").replace(\"gate_\", \"score_\")\n            lf = work / \"log\" / f\"{sub}.log\"\n            if lf.is_file():\n                tail = lf.read_text(encoding=\"utf-8\", errors=\"replace\").splitlines()[-3:]\n                print(f\"지금 단계 {name} 로그 끝:\")\n                for x in tail:\n                    print(\"   \", x[:300])\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    sub = ap.add_subparsers(dest=\"cmd\", required=True)\n    r = sub.add_parser(\"run\")\n    r.add_argument(\"--work\", default=\"~/ssaplash/cymrole_audio\")\n    r.add_argument(\"--results\", default=\"~/ssaplash/results\")\n    r.add_argument(\"--gpu\", type=int, default=6)\n    r.add_argument(\"--uv\", default=\"uv\", help=\"uv 실행 명령. 예: 'python -m uv'\")\n    r.add_argument(\"--models\", nargs=\"+\", default=REAL_MODELS, choices=list(MODELS))\n    r.add_argument(\n        \"--min-disk-gb\",\n        type=float,\n        default=150.0,\n        help=\"이보다 적게 남으면 끝난 모델 가중치를 지움\",\n    )\n    r.add_argument(\"--no-full\", action=\"store_true\", help=\"작은 시험까지만\")\n    r.add_argument(\n        \"--python\",\n        default=None,\n        help=\"로컬 시험용. 가상환경·저장소·가중치 단계를 건너뛰고 이 파이썬으로 실행\",\n    )\n    s = sub.add_parser(\"status\")\n    s.add_argument(\"--work\", default=\"~/ssaplash/cymrole_audio\")\n    args = ap.parse_args()\n    if args.cmd == \"status\":\n        status(Path(args.work).expanduser().resolve())\n        return 0\n    ctx = Ctx(args)\n    (ctx.work / \"pipeline.pid\").write_text(str(os.getpid()), encoding=\"utf-8\")\n    log(f\"작업 폴더 {ctx.work}, GPU {ctx.a.gpu}, 모델 {ctx.a.models}\")\n    if not stage(ctx, \"check\", lambda: check(ctx)):\n        return 1\n    for m in ctx.a.models:\n        run_model(ctx, m)\n    pack(ctx)\n    failed = sorted(p.name for p in ctx.failed.iterdir())\n    log(f\"전체 끝. 실패 단계: {failed or '없음'}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
}

assert (CA / "cases.jsonl").is_file(), f"{CA}에 11/09 작업 폴더가 없음"
for name, text in FILES.items():
    (CA / "scripts" / name).write_text(text, encoding="utf-8")
print("스크립트 3개 씀:", ", ".join(FILES))

uv = f"{sys.executable} -m uv"
if subprocess.run([sys.executable, "-m", "uv", "--version"], capture_output=True).returncode != 0:
    uv = "uv"                                  # 커널 파이썬에 uv가 없으면 시스템 uv

# 도는 중인지: 그 번호 프로세스가 좀비(Z)가 아니고 명령줄에 파이프라인 이름이 있을 때만.
# 11/09 실행이 끝난 뒤 좀비로 남아 번호만 보고는 '이미 도는 중'으로 잘못 나왔음
def pipeline_running(pid: int) -> bool:
    try:
        state = Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[0]
        cmd = Path(f"/proc/{pid}/cmdline").read_bytes()
    except OSError:
        return False
    return state != "Z" and b"cymrole_audio_pipeline" in cmd


pid_file = CA / "pipeline.pid"
running = pid_file.exists() and pipeline_running(int(pid_file.read_text()))
if running:
    print("이미 도는 중. pid", pid_file.read_text())
else:
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    cmd = [sys.executable, str(CA / "scripts" / "cymrole_audio_pipeline.py"), "run", "--work", str(CA),
           "--results", str(WORK / "results"), "--gpu", GPU_ID, "--uv", uv, "--models", "qwen3omni"]
    proc = subprocess.Popen(["nohup", "nice", "-n", "10", *cmd], cwd=CA, env=env,
                            stdout=open(CA / "pipeline.log", "a"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    print("시작. pid", proc.pid, "로그", CA / "pipeline.log")
# 확인: '스크립트 3개 씀: cymrole_audio_llm.py, cymrole_audio_eval.py, cymrole_audio_pipeline.py'와
#       '시작. pid 숫자 로그 /home/.../ssaplash/cymrole_audio/pipeline.log' 두 줄
# - 'cymrole_audio에 11/09 작업 폴더가 없음'이 나오면 서버 폴더가 지워진 것. 알려 줌


In [ ]:
# 2. 진행 확인 (여러 번 쳐도 됨)
import sys
from pathlib import Path

CA = Path.home() / "ssaplash" / "cymrole_audio"
!{sys.executable} {CA}/scripts/cymrole_audio_pipeline.py status --work {CA}
!tail -5 {CA}/log/ask_qwen3omni.log
!nvidia-smi -i 6 --query-gpu=memory.used,memory.total,utilization.gpu --format=csv,noheader
!ls -la ~/ssaplash/results/cymrole_audio.zip
# 확인 (도는 중 예):
#   파이프라인 도는 중 (pid 12345)
#     qwen3omni   env:끝 download:끝 pilot:- gate:- full:-
#   지금 단계 pilot_qwen3omni 로그 끝: [llm] qwen3omni_mix: 50/120, 마디당 0.80초
#   ask_qwen3omni.log 끝에 '[llm] qwen3omni_mix: 10/120, 마디당 N초'가 보이면 판정이 도는 것
# - 1~2분 안에 로그에 '불러오기 N초', 'qwen3omni 생성 결과 GenerateDecoderOnlyOutput, 점수 있음', 'qwen3omni_mix: 10/120'이 차례로 보임
# - 끝: '멈춤 또는 끝'과 로그 끝 '전체 끝'이 보이고 zip 시각이 오늘이면 끝. zip을 내려받아 로컬 Downloads에 둠
# - qwen3omni 줄에 '실패'가 보이면 위 ask_qwen3omni.log 다섯 줄을 그대로 알려 줌
